# 기준 모델 만들기: YOLO medium 4종 비교 (pegstar)

1. 실험 결과(`outputs/experiment_log.jsonl`)로 **성능 비교 막대그래프**
2. 같은 검증 이미지 1장에 **모델별 예측**을 그려 비교

- 커널 `.venv` → **Run All**
- 그림은 `reports/figures/`에 저장된다 (공개 레포라 커밋되지 않음)
- **예측 그림에는 대회 이미지가 들어가므로 디스코드로만 공유**, PR에는 막대그래프와 숫자만
- 점수는 내 PC 예비 분할 기준 (검증 47장 중 34장이 학습과 같은 알약 조합 → 점수가 부풀려져 있음)

In [ ]:
import json
import platform
from pathlib import Path

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

import sys

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if not (ROOT / "pyproject.toml").exists():
    raise FileNotFoundError("YacssoK 프로젝트 안에서 노트북을 실행해주세요.")

# scratch 폴더에서 실행해도 프로젝트의 공통 함수를 불러온다.
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils import load_config, resolve_path

CONFIG = ROOT / "configs" / "model_compare.yaml"
cfg = load_config(CONFIG)

# 데이터 위치는 YAML을 따라가도록 한다.
IMG = resolve_path(cfg["data"]["train_images"])
ANN = resolve_path(cfg["data"]["train_annotations"])

OUT = ROOT / "outputs"
FIG = ROOT / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

# 비교할 실험 (표시 이름 → 실험 이름)
comparison = cfg["comparison"]
EXPS = comparison["experiments"]

## 1. 성능 비교 막대그래프

In [ ]:
log = [json.loads(line) for line in open(OUT / "experiment_log.jsonl", encoding="utf-8")]
latest = {r["exp_name"]: r for r in log}  # 같은 이름이면 마지막 기록
rows = []
for name, exp in EXPS.items():
    r = latest.get(exp)
    if r is None:
        print("기록 없음:", exp)
        continue
    rows.append(
        {
            "모델": name,
            "mAP@[0.75:0.95]": r["map"],
            "mAP75": r["map_75"],
            "학습 시간(분)": r["train_min"],
            "epoch": r["epochs"],
            "img_size": r["img_size"],
        }
    )
res = pd.DataFrame(rows).sort_values("mAP@[0.75:0.95]", ascending=False).reset_index(drop=True)
res.index = res.index + 1
res

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [2, 1]})

# 왼쪽: 점수 (0부터 시작하는 막대, 값은 막대 위에 표시)
ax = axes[0]
x = range(len(res))
w = 0.38
b1 = ax.bar(
    [i - w / 2 for i in x],
    res["mAP@[0.75:0.95]"],
    w,
    label="mAP@[0.75:0.95] (대회 지표)",
    color="#2a6fdb",
)
b2 = ax.bar([i + w / 2 for i in x], res["mAP75"], w, label="mAP75", color="#9db8e8")
for bars in (b1, b2):
    for b in bars:
        ax.text(
            b.get_x() + b.get_width() / 2,
            b.get_height() + 0.01,
            f"{b.get_height():.3f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )
ax.set_xticks(list(x), res["모델"])
ax.set_ylim(0, 1.18)
ax.set_ylabel("검증 점수")
ax.set_title("YOLO medium 4종 검증 점수 (예비 분할, 47장)")
ax.legend(loc="upper center", ncol=2, frameon=False)
ax.spines[["top", "right"]].set_visible(False)

# 오른쪽: 학습 시간
ax = axes[1]
bars = ax.bar(res["모델"], res["학습 시간(분)"], color="#8a8f98")
for b in bars:
    ax.text(
        b.get_x() + b.get_width() / 2,
        b.get_height() + 0.2,
        f"{b.get_height():.1f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )
ax.set_ylabel("분")
ax.set_title("학습 시간 (50 epoch, RTX 5060)")
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(FIG / "mcompare_bar.png", dpi=150, bbox_inches="tight")
print("저장:", FIG / "mcompare_bar.png")

In [ ]:
# PR 본문에 붙일 표 (숫자만)
cols = ["모델", "mAP@[0.75:0.95]", "mAP75", "학습 시간(분)"]
print("| " + " | ".join(cols) + " |")
print("| " + " | ".join(["---"] * len(cols)) + " |")
for _, r in res.iterrows():
    print(
        f"| {r['모델']} | {r['mAP@[0.75:0.95]']:.3f} | {r['mAP75']:.3f} | {r['학습 시간(분)']:.1f} |"
    )

## 2. 같은 검증 이미지에 모델별 예측 그리기
모델끼리 결과가 가장 다른 이미지를 자동으로 고른다. 초록 = 정답, 빨강 = 예측(신뢰도 0.25 이상, 최대 4개). 예측 이름이 정답과 다르면 이름에 `✗`.

In [ ]:
# 평가 기준과 별도로, 비교 그림에서 낮은 신뢰도 박스를 숨긴다.
CONF = comparison["conf_threshold"]
MAX_PREDICTIONS = comparison["max_predictions"]
MATCH_IOU = comparison["match_iou"]

class_map_path = resolve_path(cfg["data"]["class_map"])
with class_map_path.open(encoding="utf-8") as f:
    class_map = json.load(f)

classes = {c["label_idx"]: c["name"] for c in class_map}
cat2name = {c["category_id"]: c["name"] for c in class_map}

preds = {}
for name, exp in EXPS.items():
    p = OUT / exp / "preds_val.json"
    if p.exists():
        preds[name] = {d["file_name"]: d for d in json.load(open(p, encoding="utf-8"))}


def gt_boxes(fname):
    boxes = []
    for jp in ANN.rglob(Path(fname).stem + ".json"):
        d = json.loads(jp.read_text(encoding="utf-8"))
        for a in d["annotations"]:
            x, y, w, h = a["bbox"]
            boxes.append(
                ([x, y, x + w, y + h], cat2name.get(a["category_id"], str(a["category_id"])))
            )
    return list({(tuple(b), n): (b, n) for b, n in boxes}.values())


def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter + 1e-9)


def kept(d):
    return [
        (b, classes.get(lab, str(lab)), s)
        for b, lab, s in zip(d["boxes"], d["labels"], d["scores"])
        if s >= CONF
    ][:MAX_PREDICTIONS]


def correct(fname, model):
    """같은 정답을 중복 집계하지 않도록 예측과 정답을 일대일로 연결한다."""
    gts = gt_boxes(fname)
    matched = set()
    count = 0

    # 신뢰도가 높은 예측부터 정답과 연결한다.
    predictions = sorted(
        kept(preds[model][fname]),
        key=lambda pred: pred[2],
        reverse=True,
    )

    for box, name, _ in predictions:
        candidates = [
            (iou(box, gt_box), index)
            for index, (gt_box, gt_name) in enumerate(gts)
            if index not in matched and name == gt_name
        ]
        if not candidates:
            continue

        best_iou, best_index = max(candidates)
        if best_iou >= MATCH_IOU:
            matched.add(best_index)
            count += 1

    return count

common = set.intersection(*(set(p) for p in preds.values()))
score = {f: [correct(f, m) for m in preds] for f in sorted(common)}
pick = max(score, key=lambda f: (max(score[f]) - min(score[f]), f))
print("고른 이미지:", pick)
print("모델별 정답 박스 수 (IoU 0.75 이상 + 이름 일치):", dict(zip(preds, score[pick])))

In [ ]:
def draw(ax, fname, model):
    im = Image.open(IMG / fname).convert("RGB")
    ax.imshow(im)
    gts = gt_boxes(fname)
    for b, _n in gts:
        ax.add_patch(
            patches.Rectangle(
                (b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec="#22c55e", lw=2.5
            )
        )
    gt_names = {n for _, n in gts}
    for b, n, s in kept(preds[model][fname]):
        ax.add_patch(
            patches.Rectangle(
                (b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec="#ef4444", lw=1.8, ls="--"
            )
        )
        mark = "" if n in gt_names else " ✗"
        ax.text(
            b[0],
            b[1] - 8,
            f"{n} {s:.2f}{mark}",
            color="white",
            fontsize=7.5,
            bbox=dict(facecolor="#ef4444", alpha=0.8, pad=1.5, lw=0),
        )
    ok = correct(fname, model)
    ax.set_title(f"{model}  (정답 {ok}/{len(gts)})", fontsize=11)
    ax.axis("off")


models = list(preds)
fig, axes = plt.subplots(1, len(models), figsize=(4.2 * len(models), 6))
for ax, m in zip(axes if len(models) > 1 else [axes], models):
    draw(ax, pick, m)
fig.suptitle(
    f"같은 검증 이미지의 모델별 예측 (초록 정답 / 빨강 예측, 신뢰도 {CONF} 이상)", fontsize=12
)
plt.tight_layout()
plt.savefig(FIG / "mcompare_pred.png", dpi=130, bbox_inches="tight")
print("저장:", FIG / "mcompare_pred.png", "→ 디스코드로만 공유")

In [ ]:
# 다른 이미지로 보고 싶으면 파일 이름을 바꿔서 실행
# pick = "K-001900-016548-019607-029451_0_2_0_2_75_000_200.png"
# 이미지별 모델 정답 수 (차이가 큰 순)
pd.DataFrame(score, index=list(preds)).T.assign(
    차이=lambda t: t.max(axis=1) - t.min(axis=1)
).sort_values("차이", ascending=False).head(10)